Modern Transformer block (Llama-style)

Pre-RMSNorm with SwiGLU FFN, the exact pattern used by Llama 4, Mistral, and DeepSeek

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# RMSNorm - simpler and faster than LayerNorm
# Used by: Llama 4, Mistral, DeepSeek V3, Gemma, Qwen
# ============================================================

class RMSNorm(nn.Module):
    """RMSNorm: scale by root-mean-square, no mean subtraction."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        # rsqrt = 1/sqrt (faster than sqrt + divide)
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

# ============================================================
# SwiGLU FFN - Gated Linear Unit with Swish activation
# Stores 2/3 of all model parameters (knowledge bank)
# ============================================================

class SwiGLU(nn.Module):
    """SwiGLU(x) = W_down(SiLU(W_gate @ x) * W_up @ x)"""
    def __init__(self, hidden_dim, intermediate_dim):
        super().__init__()
        # No bias in modern LLMs - saves params, works fine
        self.w_gate = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.w_up = nn.Linear(hidden_dim, intermediate_dim, bias=False)
        self.w_down = nn.Linear(intermediate_dim, hidden_dim, bias=False)

    def forward(self, x):
        gate = F.silu(self.w_gate(x))  # Which knowledge paths?
        up = self.w_up(x)               # Raw values
        return self.w_down(gate * up)   # Gated output

# ============================================================
# Modern Transformer Block (Pre-RMSNorm + Residual)
# x = x + Attn(RMSNorm(x)); x = x + FFN(RMSNorm(x))
# ============================================================

class ModernTransformerBlock(nn.Module):
    """Llama-style decoder block with pre-norm and SwiGLU."""
    def __init__(self, hidden_dim=4096, num_heads=32,
                 intermediate_dim=14336):
        super().__init__()
        self.norm1 = RMSNorm(hidden_dim)
        self.attn = nn.MultiheadAttention(
            hidden_dim, num_heads, batch_first=True
        )
        self.norm2 = RMSNorm(hidden_dim)
        self.ffn = SwiGLU(hidden_dim, intermediate_dim)

    def forward(self, x, mask=None):
        # Pre-norm + attention + residual
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=mask)
        x = x + attn_out  # Residual: the all-important '+x'

        # Pre-norm + SwiGLU FFN + residual
        x = x + self.ffn(self.norm2(x))
        return x

# Test it
block = ModernTransformerBlock(
    hidden_dim=512, num_heads=8, intermediate_dim=2048
)
x = torch.randn(2, 10, 512)
output = block(x)

total = sum(p.numel() for p in block.parameters())
attn_p = sum(p.numel() for p in block.attn.parameters())
ffn_p = sum(p.numel() for p in block.ffn.parameters())

print(f"Input:  {x.shape} -> Output: {output.shape}")
print(f"Total params:  {total:,}")
print(f"  Attention:   {attn_p:,} ({100*attn_p//total}%)")
print(f"  FFN (SwiGLU):{ffn_p:,} ({100*ffn_p//total}%)")

Input:  torch.Size([2, 10, 512]) -> Output: torch.Size([2, 10, 512])
Total params:  4,197,376
  Attention:   1,050,624 (25%)
  FFN (SwiGLU):3,145,728 (74%)
